In [ ]:
!pip install -U chronos-forecasting -q

In [ ]:
import numpy as np
import torch
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from chronos import Chronos2Pipeline

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
)
print('Chronos-2 loaded')

def get_chronos2_embeddings(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        context = torch.tensor(batch, dtype=torch.float32)
        embeddings, _ = pipeline.embed(context)
        for emb in embeddings:
            pooled = emb.mean(dim=1).flatten().numpy()
            all_embeddings.append(pooled)
        print(f'  {min(i+batch_size, len(X))}/{len(X)} done', flush=True)
    return np.array(all_embeddings)

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])
y_tr = np.array([int(x) for x in data['train']['y']])
print(X_te.shape, y_tr.shape, y_te.shape)

In [ ]:
rng = np.random.default_rng(0)
T = X_te.shape[2]

def desync(X):
    out = np.empty_like(X)
    for i in range(len(X)):
        for c in range(X.shape[1]):
            out[i, c] = np.roll(X[i, c], rng.integers(-T//4, T//4))
    return out

def global_roll(X):
    out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = np.roll(X[i], rng.integers(-T//4, T//4), axis=1)
    return out

Z_tr = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_train_emb.npy')
Z_te = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_test_emb.npy')
print('loaded emb:', Z_tr.shape, Z_te.shape)

clf = RidgeClassifier().fit(Z_tr, y_tr)
print(f'intact                  acc {accuracy_score(y_te, clf.predict(Z_te)):.4f}')

for tag, Xv, fname in [
    ('global roll (aligned)', global_roll(X_te), 'Chronos2_CHBMIT_v2_globalroll_test_emb.npy'),
    ('desync (misaligned)',   desync(X_te),      'Chronos2_CHBMIT_v2_desync2_test_emb.npy'),
]:
    Z = get_chronos2_embeddings(pipeline, Xv)
    np.save(f'{SAVE_DIR}/{fname}', Z)
    print(f'{tag:23} acc {accuracy_score(y_te, clf.predict(Z)):.4f}')

In [ ]:
import os
clf = RidgeClassifier().fit(Z_tr, y_tr)

files = [
 ('intact',            'Chronos2_CHBMIT_v2_test_emb.npy'),
 ('permuted (saved)',  'Chronos2_CHBMIT_v2_permuted_test_emb.npy'),
 ('desync (saved)',    'Chronos2_CHBMIT_v2_desync_test_emb.npy'),
 ('pooled (saved)',    'Chronos2_CHBMIT_v2_chanpool_test_emb.npy'),
 ('global roll (new)', 'Chronos2_CHBMIT_v2_globalroll_test_emb.npy'),
 ('desync (new)',      'Chronos2_CHBMIT_v2_desync2_test_emb.npy'),
]
for tag, f in files:
    p = f'{SAVE_DIR}/{f}'
    if not os.path.exists(p):
        print(f'{tag:20} MISSING'); continue
    Z = np.load(p)
    if Z.shape[1] != Z_tr.shape[1]:
        print(f'{tag:20} dim {Z.shape[1]} != probe {Z_tr.shape[1]}, needs own probe'); continue
    print(f'{tag:20} acc {accuracy_score(y_te, clf.predict(Z)):.4f}')

In [ ]:
Ztr_p = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_chanpool_train_emb.npy')
Zte_p = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_chanpool_test_emb.npy')
clf_p = RidgeClassifier().fit(Ztr_p, y_tr)
print(f'pooled (own probe)   dim {Ztr_p.shape[1]}   acc {accuracy_score(y_te, clf_p.predict(Zte_p)):.4f}')

In [ ]:
rng2 = np.random.default_rng(1)
D = 768
n_ch = Z_te.shape[1] // D
print('blocks:', n_ch, 'x', D)

accs = []
for trial in range(5):
    perm = rng2.permutation(n_ch)
    Zp = np.concatenate([Z_te[:, p*D:(p+1)*D] for p in perm], axis=1)
    accs.append(accuracy_score(y_te, clf.predict(Zp)))
print('embedding-block permutation (no model re-run):',
      [f'{a:.4f}' for a in accs], f'mean {np.mean(accs):.4f}')
print('input-channel permutation (model re-run):      0.7069')
print('intact:                                        0.8027')

In [ ]:
from sklearn.random_projection import GaussianRandomProjection

print(f'{"intact (13824d)":34} acc {accuracy_score(y_te, clf.predict(Z_te)):.4f}')
print(f'{"pooled (768d, own probe)":34} acc 0.8736')
print()

for seed in range(3):
    rp = GaussianRandomProjection(n_components=768, random_state=seed).fit(Z_tr)
    c = RidgeClassifier().fit(rp.transform(Z_tr), y_tr)
    a = accuracy_score(y_te, c.predict(rp.transform(Z_te)))
    print(f'  intact projected to 768d, seed {seed}: acc {a:.4f}')

print()
for i in range(0, 18, 6):
    c = RidgeClassifier().fit(Z_tr[:, i*768:(i+1)*768], y_tr)
    a = accuracy_score(y_te, c.predict(Z_te[:, i*768:(i+1)*768]))
    print(f'  single channel {i} block only (768d): acc {a:.4f}')

In [ ]:
import glob, os
print([os.path.basename(p) for p in sorted(glob.glob(f'{SAVE_DIR}/*MOMENT*'))])
print([os.path.basename(p) for p in sorted(glob.glob(f'{SAVE_DIR}/*Mantis*'))])

In [ ]:
for tag in ['MantisV2_CHBMIT', 'MantisV2_CHBMIT_v2']:
    a = np.load(f'{SAVE_DIR}/{tag}_train_emb.npy')
    b = np.load(f'{SAVE_DIR}/{tag}_test_emb.npy')
    c = RidgeClassifier().fit(a, y_tr)
    print(f'{tag:24} dim {a.shape[1]:>6}  acc {accuracy_score(y_te, c.predict(b)):.4f}')

In [ ]:
from sklearn.linear_model import RidgeClassifierCV
A = np.logspace(-2, 5, 15)

def cv(Ztr, Zte, tag):
    c = RidgeClassifierCV(alphas=A).fit(Ztr, y_tr)
    print(f'{tag:30} acc {accuracy_score(y_te, c.predict(Zte)):.4f}  alpha {c.alpha_:.1f}')
    return c

c_int = cv(Z_tr, Z_te, 'Chronos-2 intact 13824d')
for tag, f in [('permuted','permuted'), ('desync','desync'), ('desync2','desync2'), ('globalroll','globalroll')]:
    Z = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_{f}_test_emb.npy')
    print(f'  {tag:28} acc {accuracy_score(y_te, c_int.predict(Z)):.4f}')

cv(Ztr_p, Zte_p, 'Chronos-2 pooled 768d')
cv(np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_train_emb.npy'),
   np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_test_emb.npy'), 'MantisV2 v2 4608d')

In [ ]:
from sklearn.linear_model import RidgeClassifierCV
A = np.logspace(-2, 5, 15)
MERGE = 30
file_te = np.asarray(data['test']['file']); t0_te = data['test']['t0'].astype(int)

event_id = np.full(len(y_te), -1, int); nxt = 0
for f in np.unique(file_te):
    idx = np.where((file_te == f) & (y_te == 1))[0]
    if not len(idx): continue
    o = idx[np.argsort(t0_te[idx])]; s = t0_te[o]
    loc = np.zeros(len(o), int)
    for i in range(1, len(o)): loc[i] = loc[i-1] + (1 if s[i]-s[i-1] > MERGE else 0)
    for l, g in zip(loc, o): event_id[g] = nxt + l
    nxt += int(loc.max())+1
neg = (y_te == 0); n_ev = nxt

def sens(prob, tgt=0.05):
    best = 0
    for thr in np.unique(prob):
        p = prob >= thr
        if p[neg].mean() > tgt: continue
        d = np.mean([p[event_id==e].any() for e in range(n_ev)])
        best = max(best, d)
    return best

for tag, tr, te in [
  ('Chronos-2', Z_tr, Z_te),
  ('MantisV2', np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_train_emb.npy'),
               np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_test_emb.npy'))]:
    c = RidgeClassifierCV(alphas=A).fit(tr, y_tr)
    print(f'{tag:12} sens@5%FPR {sens(c.decision_function(te)):.3f}  ({n_ev} events)')

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

Z_tr = get_chronos2_embeddings(pipeline, X_tr)
Z_te = get_chronos2_embeddings(pipeline, X_te)

clf = RidgeClassifier()
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'Chronos-2 Accuracy on CHBMIT v2: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_test_emb.npy', Z_te)